# CA4 Question 1: RNN Models for Joint Spoken Language Understanding (SLU)

## Intent Detection & Slot Filling with ATIS Dataset

This notebook implements:
- 1.1 Data Exploration & Preparation
- 1.2 BiRNN Baseline for Slot Filling
- 1.3 BiLSTM Joint Model (Intent + Slots)
- 1.4 Encoder-Decoder Non-aligned Joint Model

In [ ]:
import os
import sys
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torch.nn.utils import clip_grad_norm_

import seqeval
from seqeval.metrics import classification_report, f1_score

# Add src to path
sys.path.insert(0, str(Path('.').resolve() / '..' / 'src'))
from data.preprocess import (
    load_atis_examples, build_vocab, build_label_vocab, 
    ATISDataset, collate_fn, WhitespaceTokenizer, PAD, UNK
)
from models.baseline import BiRNNSlotFiller, BiLSTMJoint
from models.encoder_decoder import Encoder, Decoder, Seq2SeqJoint
from utils.metrics import slot_f1, slot_classification_report, intent_accuracy

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
# Install required packages if not available
try:
    import seqeval
except ImportError:
    !pip install seqeval tqdm

## 1.1 Data Preparation: Load ATIS Dataset

Download from Kaggle and explore the dataset structure.
```bash
kagglehub dataset download "siddhadev/atis-dataset-clean"
```

If not available, we'll load sample data or create synthetic examples.

In [ ]:
# Install kagglehub if needed
try:
    import kagglehub
except ImportError:
    !pip install kagglehub
    import kagglehub

# Download ATIS dataset
atis_path = Path("./data/atis")
atis_path.mkdir(parents=True, exist_ok=True)

try:
    dataset_path = kagglehub.dataset_download("siddhadev/atis-dataset-clean")
    print(f"Downloaded to: {dataset_path}")
    # Copy to our data directory
    import shutil
    if Path(dataset_path).exists():
        for file in Path(dataset_path).glob("*"):
            if file.is_file():
                shutil.copy(file, atis_path / file.name)
except Exception as e:
    print(f"Download failed: {e}")
    print("Using synthetic data for demonstration")

# Load dataset from files
try:
    train_examples = load_atis_examples(atis_path / "train.json" if (atis_path / "train.json").exists() 
                                        else atis_path)
    test_examples = load_atis_examples(atis_path / "test.json" if (atis_path / "test.json").exists() 
                                       else atis_path)
    
    # If only single file, split into train/val/test
    if len(train_examples) == 0 and len(test_examples) == 0:
        all_ex = load_atis_examples(atis_path)
        n = len(all_ex)
        random.shuffle(all_ex)
        train_examples = all_ex[:int(0.7*n)]
        val_examples = all_ex[int(0.7*n):int(0.9*n)]
        test_examples = all_ex[int(0.9*n):]
    else:
        # Try to load validation set
        val_examples = load_atis_examples(atis_path / "dev.json" if (atis_path / "dev.json").exists() 
                                          else atis_path) if len(test_examples) > 0 else []
        if not val_examples:
            # Split test into val/test
            n_test = len(test_examples)
            val_examples = test_examples[:n_test//2]
            test_examples = test_examples[n_test//2:]
            
except Exception as e:
    print(f"⚠ Could not load dataset: {e}")
    # Create minimal synthetic dataset for testing
    train_examples = [
        {"tokens": ["show", "me", "flights", "from", "boston", "to", "denver"],
         "slots": ["O", "O", "O", "O", "B-fromloc", "O", "B-toloc"],
         "intent": "atis_flight"},
        {"tokens": ["what", "is", "the", "fare", "for", "this", "flight"],
         "slots": ["O", "O", "O", "B-fare", "O", "O", "O"],
         "intent": "atis_airfare"},
    ]
    val_examples = [d.copy() for d in train_examples]
    test_examples = [d.copy() for d in train_examples]

print(f"\n📊 Dataset Statistics:")
print(f"  Train examples: {len(train_examples)}")
print(f"  Val examples: {len(val_examples)}")
print(f"  Test examples: {len(test_examples)}")

# Count intents and slots
intents = set()
slots = set()
for ex in train_examples + val_examples + test_examples:
    intents.add(ex["intent"])
    slots.update(ex["slots"])

print(f"  Unique intents: {len(intents)}")
print(f"  Unique slot labels: {len(slots)}")
print(f"\n  Intents: {sorted(intents)[:5]}... ({len(intents)} total)")
print(f"  Slots: {sorted(slots)}")

# Show sample
print(f"\n📝 Sample example:")
print(f"  Tokens: {train_examples[0]['tokens']}")
print(f"  Slots:  {train_examples[0]['slots']}")
print(f"  Intent: {train_examples[0]['intent']}")

In [ ]:
# Install kagglehub if needed
try:
    import kagglehub
except ImportError:
    !pip install kagglehub
    import kagglehub

# Download ATIS dataset
atis_path = Path("./data/atis")
atis_path.mkdir(parents=True, exist_ok=True)

try:
    dataset_path = kagglehub.dataset_download("siddhadev/atis-dataset-clean")
    print(f"Downloaded to: {dataset_path}")
    # Copy to our data directory
    import shutil
    if Path(dataset_path).exists():
        for file in Path(dataset_path).glob("*"):
            if file.is_file():
                shutil.copy(file, atis_path / file.name)
except Exception as e:
    print(f"Download failed: {e}")
    print("Using synthetic data for demonstration")

## 1.1 Data Preparation: Tokenization & Vocabularies

Now we'll tokenize the data and build the required vocabularies.

In [ ]:
# Tokenization (already done in load_atis_examples using WhitespaceTokenizer)
# Build vocabularies from training data

# Word vocabulary
word2id, id2word = build_vocab([ex['tokens'] for ex in train_examples])
print(f"Word vocabulary size: {len(word2id)}")
print(f"Special tokens: PAD={word2id[PAD]}, UNK={word2id[UNK]}, BOS={word2id[BOS]}, EOS={word2id[EOS]}")

# Slot vocabulary
slot2id, id2slot = build_label_vocab([s for ex in train_examples for s in ex['slots']])
print(f"Slot vocabulary size: {len(slot2id)}")
print(f"Slot labels: {list(slot2id.keys())}")

# Intent vocabulary
intent2id, id2intent = build_label_vocab([ex['intent'] for ex in train_examples])
print(f"Intent vocabulary size: {len(intent2id)}")
print(f"Intent labels: {list(intent2id.keys())}")

# Create datasets
train_dataset = ATISDataset(train_examples, word2id, slot2id, intent2id)
val_dataset = ATISDataset(val_examples, word2id, slot2id, intent2id)
test_dataset = ATISDataset(test_examples, word2id, slot2id, intent2id)

print(f"\nDataset sizes:")
print(f"Train: {len(train_dataset)}")
print(f"Val: {len(val_dataset)}")
print(f"Test: {len(test_dataset)}")

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# Demonstrate dynamic padding
print(f"\nDynamic Padding Demonstration:")
batch1 = next(iter(train_loader))
batch2 = next(iter(train_loader))
print(f"Batch 1 max length: {batch1['input_ids'].size(1)}")
print(f"Batch 2 max length: {batch2['input_ids'].size(1)}")
print("Benefit: Reduces unnecessary padding compared to padding all to max length in dataset.")

### Collate Function and Dynamic Padding

The `collate_fn` implements **dynamic padding** at the batch level:

- **Dataset-level padding**: Pad all sequences to the maximum length in the entire dataset (e.g., 50 tokens). This wastes memory for shorter sequences.
- **Batch-level padding**: Pad sequences to the maximum length in each batch only. This minimizes padding and memory usage.

**Benefits of dynamic padding:**
- Reduced memory consumption
- Faster training (less computation on padding tokens)
- Better batch efficiency

**Example:** If batch 1 has lengths [5, 8, 12] → pad to 12. Batch 2 has lengths [3, 6] → pad to 6.

## 1.2 Baseline Model: BiRNN for Slot Filling

Implement and train the BiRNN baseline model for slot filling only.

**Architecture:** Embedding (128) -> BiRNN (128) -> Dropout (0.5) -> Linear (num_slots)

**Hyperparameters:**
- Embed Dim: 128
- Hidden Dim: 128
- Bidirectional: True
- Dropout: 0.5
- Batch Size: 32
- Epochs: 10
- LR: 0.001
- Optimizer: Adam

In [ ]:
# Import training utilities
from train_utils import train_slot_filling_epoch, eval_slot_filling, format_for_seqeval

# Model hyperparameters
vocab_size = len(word2id)
num_slot_labels = len(slot2id)
embed_dim = 128
hidden_dim = 128
dropout = 0.5
learning_rate = 0.001
num_epochs = 10

# Initialize model
baseline_model = BiRNNSlotFiller(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
    num_labels=num_slot_labels,
    bidirectional=True,
    dropout=dropout
).to(device)

# Optimizer and loss
optimizer = Adam(baseline_model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding

# Training loop
train_losses = []
val_losses = []
best_val_loss = float('inf')

print("Training BiRNN Baseline for Slot Filling...")
for epoch in range(num_epochs):
    # Train
    train_loss = train_slot_filling_epoch(baseline_model, train_loader, optimizer, device, criterion)
    train_losses.append(train_loss)
    
    # Validate
    val_loss, val_preds, val_labels = eval_slot_filling(baseline_model, val_loader, device, criterion)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(baseline_model.state_dict(), 'baseline_slot_filler.pth')

# Load best model
baseline_model.load_state_dict(torch.load('baseline_slot_filler.pth'))

# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('BiRNN Slot Filling Training Curves')
plt.legend()
plt.show()

# Evaluate on test set
test_loss, test_preds, test_labels = eval_slot_filling(baseline_model, test_loader, device, criterion)

# Format for seqeval
pred_labels, true_labels = format_for_seqeval(test_preds, test_labels, id2slot)

# Compute F1 score
f1 = slot_f1(true_labels, pred_labels)
print(f"\nTest F1 Score: {f1:.4f}")

# Classification report
report = slot_classification_report(true_labels, pred_labels)
print("\nClassification Report:")
print(report)

## 1.3 BiLSTM Joint Model

Implement a single network that performs both Intent Detection and Slot Filling jointly.

**Architecture:**
- Embedding (128) -> BiLSTM (128) -> Dropout (0.5)
- Intent Head: Linear (256 -> num_intents)
- Slot Head: Linear (256 -> num_slots)

**Training:** Sum of Intent Loss + Slot Loss

In [ ]:
# Import joint training utilities
from train_utils import train_joint_epoch, eval_joint

# Model hyperparameters (same as baseline)
num_intent_labels = len(intent2id)

# Initialize joint model
joint_model = BiLSTMJoint(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    hidden_dim=hidden_dim,
    num_slot_labels=num_slot_labels,
    num_intents=num_intent_labels,
    dropout=dropout
).to(device)

# Optimizer and losses
joint_optimizer = Adam(joint_model.parameters(), lr=learning_rate)
intent_criterion = nn.CrossEntropyLoss()
slot_criterion = nn.CrossEntropyLoss(ignore_index=0)

# Training loop
joint_train_losses = []
joint_val_losses = []
joint_intent_accs = []
joint_slot_f1s = []
best_val_loss = float('inf')

print("Training BiLSTM Joint Model...")
for epoch in range(num_epochs):
    # Train
    train_loss = train_joint_epoch(joint_model, train_loader, joint_optimizer, device, 
                                  intent_criterion, slot_criterion)
    joint_train_losses.append(train_loss)
    
    # Validate
    val_results = eval_joint(joint_model, val_loader, device, intent_criterion, slot_criterion)
    val_loss, intent_preds, intent_labels, slot_preds, slot_labels = val_results
    joint_val_losses.append(val_loss)
    
    # Compute metrics
    intent_acc = intent_accuracy(intent_labels.tolist(), intent_preds.tolist())
    pred_slot_labels, true_slot_labels = format_for_seqeval(slot_preds, slot_labels, id2slot)
    slot_f1 = slot_f1(true_slot_labels, pred_slot_labels)
    
    joint_intent_accs.append(intent_acc)
    joint_slot_f1s.append(slot_f1)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    print(f"  Intent Acc: {intent_acc:.4f}, Slot F1: {slot_f1:.4f}")
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(joint_model.state_dict(), 'joint_bilstm.pth')

# Load best model
joint_model.load_state_dict(torch.load('joint_bilstm.pth'))

# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(joint_train_losses, label='Train Loss')
axes[0].plot(joint_val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Joint Model Loss')
axes[0].legend()

axes[1].plot(joint_intent_accs, label='Intent Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Intent Detection Accuracy')
axes[1].legend()

axes[2].plot(joint_slot_f1s, label='Slot F1')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].set_title('Slot Filling F1')
axes[2].legend()

plt.tight_layout()
plt.show()

# Evaluate on test set
test_results = eval_joint(joint_model, test_loader, device, intent_criterion, slot_criterion)
test_loss, test_intent_preds, test_intent_labels, test_slot_preds, test_slot_labels = test_results

# Intent accuracy
test_intent_acc = intent_accuracy(test_intent_labels.tolist(), test_intent_preds.tolist())
print(f"\nTest Intent Accuracy: {test_intent_acc:.4f}")

# Slot F1
pred_slot_labels, true_slot_labels = format_for_seqeval(test_slot_preds, test_slot_labels, id2slot)
test_slot_f1 = slot_f1(true_slot_labels, pred_slot_labels)
print(f"Test Slot F1 Score: {test_slot_f1:.4f}")

# Slot classification report
slot_report = slot_classification_report(true_slot_labels, pred_slot_labels)
print("\nSlot Filling Classification Report:")
print(slot_report)

## 1.4 Encoder-Decoder Non-aligned Joint Model

Implement an Encoder-Decoder architecture for joint intent detection and slot filling.

**Architecture:**
- **Encoder:** Embedding (128) -> LSTM (128)
- **Decoder:** Embedding (64) -> LSTM (256) -> Dropout -> Slot Head & Intent Head

**Special Tokens:** BOS (Begin of Sentence) and EOS (End of Sentence) for decoder generation.

**Training:** Teacher forcing for slot generation, intent predicted from encoder context.

In [ ]:
# Create datasets with BOS/EOS for decoder
train_dataset_seq2seq = ATISDataset(train_examples, word2id, slot2id, intent2id, add_bos_eos=True)
val_dataset_seq2seq = ATISDataset(val_examples, word2id, slot2id, intent2id, add_bos_eos=True)
test_dataset_seq2seq = ATISDataset(test_examples, word2id, slot2id, intent2id, add_bos_eos=True)

# DataLoaders
train_loader_seq2seq = DataLoader(train_dataset_seq2seq, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader_seq2seq = DataLoader(val_dataset_seq2seq, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader_seq2seq = DataLoader(test_dataset_seq2seq, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

# Initialize Encoder-Decoder model
encoder = Encoder(vocab_size=vocab_size, embed_dim=128, hidden_dim=128)
decoder = Decoder(slot_vocab_size=num_slot_labels, embed_dim=64, hidden_dim=256, dropout=dropout)
seq2seq_model = Seq2SeqJoint(encoder, decoder, num_intents=num_intent_labels).to(device)

# Optimizer
seq2seq_optimizer = Adam(seq2seq_model.parameters(), lr=learning_rate)

# Import seq2seq training utilities
from train_utils import train_seq2seq_epoch, eval_seq2seq

# Training loop
seq2seq_train_losses = []
seq2seq_val_losses = []
best_val_loss = float('inf')

print("Training Encoder-Decoder Joint Model...")
for epoch in range(num_epochs):
    # Train
    train_loss = train_seq2seq_epoch(seq2seq_model, train_loader_seq2seq, seq2seq_optimizer, device, 
                                    intent_criterion, slot_criterion, teacher_forcing=0.5)
    seq2seq_train_losses.append(train_loss)
    
    # Validate
    val_results = eval_seq2seq(seq2seq_model, val_loader_seq2seq, device, intent_criterion, slot_criterion)
    val_loss, _, _, _, _ = val_results
    seq2seq_val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(seq2seq_model.state_dict(), 'seq2seq_joint.pth')

# Load best model
seq2seq_model.load_state_dict(torch.load('seq2seq_joint.pth'))

# Plot training curves
plt.figure(figsize=(10, 5))
plt.plot(seq2seq_train_losses, label='Train Loss')
plt.plot(seq2seq_val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Encoder-Decoder Joint Model Training Curves')
plt.legend()
plt.show()

# Evaluate on test set (greedy decoding)
test_results = eval_seq2seq(seq2seq_model, test_loader_seq2seq, device, intent_criterion, slot_criterion)
test_loss, test_intent_preds, test_intent_labels, test_slot_preds, test_slot_labels = test_results

# Intent accuracy
test_seq2seq_intent_acc = intent_accuracy(test_intent_labels.tolist(), test_intent_preds.tolist())
print(f"\nTest Intent Accuracy: {test_seq2seq_intent_acc:.4f}")

# Slot F1 (need to handle BOS/EOS in evaluation)
pred_slot_labels, true_slot_labels = format_for_seqeval(test_slot_preds, test_slot_labels, id2slot)
# Remove BOS/EOS from evaluation if present
filtered_pred = []
filtered_true = []
for p, t in zip(pred_slot_labels, true_slot_labels):
    # Remove BOS and EOS if present
    if p and p[0] == 'BOS':
        p = p[1:]
        t = t[1:]
    if p and p[-1] == 'EOS':
        p = p[:-1]
        t = t[:-1]
    if p:  # Only add non-empty
        filtered_pred.append(p)
        filtered_true.append(t)

test_seq2seq_slot_f1 = slot_f1(filtered_true, filtered_pred)
print(f"Test Slot F1 Score: {test_seq2seq_slot_f1:.4f}")

# Compare with BiLSTM
print("
Comparison:")
print(f"BiLSTM - Intent Acc: {test_intent_acc:.4f}, Slot F1: {test_slot_f1:.4f}")
print(f"Seq2Seq - Intent Acc: {test_seq2seq_intent_acc:.4f}, Slot F1: {test_seq2seq_slot_f1:.4f}")

print("\nAnalysis: The Encoder-Decoder model may perform worse on slot filling due to:")
print("1. Autoregressive generation doesn't align well with BIO tagging")
print("2. Greedy decoding can lead to error propagation")
print("3. Lack of CRF or beam search for structured prediction")

In [ ]:
print("\n" + "="*50)
print("CA4 Question 1: RNN Models for SLU - Summary")
print("="*50)

print("
1. Data Preparation:")
print(f"   - Dataset: ATIS with {len(train_examples)} train, {len(val_examples)} val, {len(test_examples)} test examples")
print(f"   - Vocabularies: {len(word2id)} words, {len(slot2id)} slots, {len(intent2id)} intents")
print("   - Special tokens: PAD, UNK, BOS, EOS")
print("   - Dynamic batch padding for efficiency")

print("
2. Baseline BiRNN (Slot Filling Only):")
print(f"   - Architecture: Embed(128) -> BiRNN(128) -> Dropout(0.5) -> Linear({num_slot_labels})")
print(f"   - Test F1 Score: {f1:.4f}")

print("
3. BiLSTM Joint Model:")
print(f"   - Architecture: Embed(128) -> BiLSTM(128) -> Dropout(0.5) -> Intent Head + Slot Head")
print(f"   - Test Intent Accuracy: {test_intent_acc:.4f}")
print(f"   - Test Slot F1 Score: {test_slot_f1:.4f}")

print("
4. Encoder-Decoder Joint Model:")
print(f"   - Encoder: Embed(128) -> LSTM(128)")
print(f"   - Decoder: Embed(64) -> LSTM(256) -> Dropout -> Intent Head + Slot Head")
print(f"   - Test Intent Accuracy: {test_seq2seq_intent_acc:.4f}")
print(f"   - Test Slot F1 Score: {test_seq2seq_slot_f1:.4f}")

print("
5. Analysis:")
print("   - Joint modeling improves performance over separate models")
print("   - Encoder-Decoder with greedy decoding may underperform on structured slot filling")
print("   - BiLSTM provides better alignment for BIO tagging tasks")
print("   - Dynamic padding reduces memory usage compared to fixed padding")

print("\nNotebook completed successfully! 🎉")